In [ ]:
%pip install pathlib
%pip install Pillow

import re
import shutil
from pathlib import Path
from PIL import Image


def main(original_dir, working_dir):
    original_path = Path(original_dir).resolve()
    working_path = Path(working_dir).resolve()

    match original_path.exists(), working_path.exists():
        case True, True:
            shutil.rmtree(working_path)
            shutil.copytree(original_path, working_path)
        case True, False:
            shutil.copytree(original_path, working_path)
        case _:
            raise ValueError("Invalid directory states")

    extensions = {".png", ".jpg", ".jpeg"}
    sizes = {"sml": 640, "med": 1280, "big": 1920}

    size_limits = {k: (v, v) for k, v in sizes.items()}
    pattern = re.compile(rf"\.({"|".join(sizes.keys())})$")

    path_list = list(working_path.rglob("*"))

    for path in path_list:
        if path.suffix in extensions:
            try:
                img = Image.open(path)
                
                limit_key = None
                
                if match := pattern.search(path.stem):
                    limit_key = match.group(1)
                    new_stem = pattern.sub("", path.stem)
                else:
                    new_stem = path.stem

                if limit_key:
                    max_size = size_limits[limit_key]
                    img.thumbnail(max_size, Image.Resampling.LANCZOS)

                webp_path = path.with_name(f"{new_stem}.webp")

                img.save(webp_path, "webp", quality=80, method=6)

                path.unlink()
                print(f"Replaced: {path.name} -> {webp_path.name}")

            except Exception as e:
                print(f"Error processing {path}: {e}")


main("./images.bak", "./images")